# 🏷️ Addendum: Type Hints in Python

Designed by Neil Ghugare.

This is an addendum to the general documenting code notebook.

___

While Python is a dynamically typed language (meaning you don't *have* to declare variable types), modern Python (3.5+) supports **Type Hints** (defined in PEP 484). 

Type annotations do not change how Python executes at runtime, but they make your code self-documenting, power rich editor autocompletion in VSCode, and allow static bug checkers (like `mypy`) to catch errors before you run your analysis pipeline.

___

## 1. Basic Function Syntax
Type hints use `:` after parameter/variable names to specify input types, and `->` after the parameter list to specify return types.

[*I've purposefully made these comments bad throughout this entire notebook, maybe as an exercise you should try to improve them :)*]

In [9]:
def format_target_summary(star_id: int, temperature: float, is_confirmed: bool = True) -> str:
    """Format an astronomical target summary string."""
    status = "Confirmed" if is_confirmed else "Candidate"
    return f"Target HD-{star_id}: {temperature:.1f} K [{status}]"

# Normal execution works as expected.
summary = format_target_summary(10283, 5778.0)
print(summary)


Target HD-10283: 5778.0 K [Confirmed]


You can use these type hints for variables themselves too:

In [10]:
# Type hints for a variable.
string: str = 'hello'

print(string)

hello


But, we can see how these type hints don't do anything for how Python actually functions. These are just for us:

In [11]:
totally_a_string: str = False

print(totally_a_string)

False


In [12]:
print(type(totally_a_string))

<class 'bool'>


From this, we can see that even though we type-hinted `totally_a_string` as a `str`, we initialized it to a `bool` value of `False`. As such, Python still treated it as a `bool` (as indicated by the `type` being retunred as a `bool`). So type-hinting is not a secure way to ensure Python treats things properly. You should be using type checks or other measures to ensure that works. This is just to make things easier on us and our IDEs. 

___

## 2. Scientific Data Structures & Union/Optional Types

In scientific computing, arguments are often more complex than standard Python primitives. You might accept either a single scalar `float` or a `numpy.ndarray`, or optional parameters that default to `None`.

You can import typing constructs from Python's built-in typing module:

In [13]:
# This `typing` important allows us to use better type hinting.
from typing import Union, Optional, List
import numpy as np

# Create a custom type alias for readability.
# This is a "union" of two possible types: a float or a np.ndarray.
ArrayOrFloat = Union[float, np.ndarray]

def calculate_redshift(
    observed_wavelength: ArrayOrFloat,  # We use the union here.
    rest_wavelength: float,
    mask: Optional[np.ndarray] = None   # We directly indicate that 'mask' is an optional parameter with 'Optional'.
) -> ArrayOrFloat:
    """
    Calculate spectral redshift z = (lambda_obs - lambda_rest) / lambda_rest.
    """
    z = (observed_wavelength - rest_wavelength) / rest_wavelength
    
    if mask is not None:
        return z[mask]
        
    return z

# Example usage with arrays.
obs_wave = np.array([6568.0, 6570.0, 6572.0])
rest_wave = 6562.8  # H-alpha in Angstroms.

z_values = calculate_redshift(observed_wavelength=obs_wave, rest_wavelength=rest_wave)
print("Calculated redshifts:", z_values)

# Example usage with singular float:
print("Singular calculated redshift at 5000A: ", 
      calculate_redshift(observed_wavelength=5000, rest_wavelength=rest_wave))


Calculated redshifts: [0.00079234 0.00109709 0.00140184]
Singular calculated redshift at 5000A:  -0.23813006643505824


___

## 3. Key Benefits of Type Hints for Scientific Code

1. **Instant IDE Autocomplete:** When you type a variable name in VS Code or PyCharm, the editor immediately suggests valid methods because it knows its exact type (e.g., suggesting `.mean()` or `.shape` for arrays).
2. **Clear API Boundaries:** Collaborators instantly see whether a function expects a single float or a `List[str]` without reading through the implementation code.
3. **Early Bug Detection:** Running static analyzers like mypy across your codebase catches subtle type mismatches (such as accidentally passing a string path into a math function) before you launch an overnight simulation cluster job.